# Nutrient Mapping — Food Name Matching

The FCD_id in your foods file (USDA FoodData Central) and the ingred_code in min_cost_data (USDA SR) are **different ID systems** — they can't be joined directly.

This notebook maps them by **food name** instead, with a manual review step to fix any bad matches.

## Step 1 — Install & Import

In [7]:
# %pip install thefuzz python-Levenshtein openpyxl --quiet

import pandas as pd
import numpy as np
from thefuzz import process
import warnings
warnings.filterwarnings('ignore')

## Step 2 — Load Data

In [8]:
# Your foods with prices
foods = pd.read_csv('foods_with_fcd_id.csv')
print('Foods shape:', foods.shape)
foods.head()

Foods shape: (176, 8)


,Food,Quantity,Units,Price,Year,FCD_id,FDC_match,FDC_type
0,white flour,1.0,pound,0.540,2023,790146,NaN,NaN
1,rice long grain,1.0,pound,0.970,2023,169756,NaN,NaN
2,spaghetti,1.0,pound,1.475,2023,2099117,NaN,NaN
3,white bread,1.0,pound,1.888,2023,2456464,NaN,NaN
4,whole wheat bread,1.0,pound,2.451,2023,2286812,NaN,NaN


In [10]:
# Nutrients from min_cost_data
nutrients = pd.read_excel('min_cost_data(1).xlsx', sheet_name='nutrients', engine='openpyxl')
print('Nutrients shape:', nutrients.shape)
nutrients.head()

Nutrients shape: (2750, 67)


,ingred_code,Ingredient description,Capric acid,Lauric acid,Myristic acid,Palmitic acid,Palmitoleic acid,Stearic acid,Oleic acid,Linoleic Acid,...,Vitamin B12,"Vitamin B-12, added",Vitamin B6,Vitamin C,Vitamin D,Vitamin E,"Vitamin E, added",Vitamin K,Water,Zinc
0,1001,"Butter, salted",2.529,2.587,7.436,21.697,0.961,9.999,19.961,2.728,...,0.17,0.0,0.003,0.0,0.0,2.32,0.0,7.0,15.87,0.09
1,1002,"Butter, whipped, with salt",2.039,2.354,7.515,20.531,1.417,7.649,17.370,2.713,...,0.07,0.0,0.008,0.0,0.0,1.37,0.0,4.6,16.72,0.05
2,1003,"Butter oil, anhydrous",2.495,2.793,10.005,26.166,2.228,12.056,25.026,2.247,...,0.01,0.0,0.001,0.0,0.0,2.80,0.0,8.6,0.24,0.01
3,1004,"Cheese, blue",0.601,0.491,3.301,9.153,0.816,3.235,6.622,0.536,...,1.22,0.0,0.166,0.0,0.5,0.25,0.0,2.4,42.41,2.66
4,1005,"Cheese, brick",0.585,0.482,3.227,8.655,0.817,3.455,7.401,0.491,...,1.26,0.0,0.065,0.0,0.5,0.26,0.0,2.5,41.11,2.60


## Step 3 — Fuzzy Match by Food Name

In [11]:
food_names     = foods['Food'].tolist()
nutrient_names = nutrients['Ingredient description'].tolist()

matches = []
for food in food_names:
    best_match, score = process.extractOne(food, nutrient_names)
    matches.append({
        'Food'       : food,
        'matched_to' : best_match,
        'score'      : score
    })

match_df = pd.DataFrame(matches)

# Save for manual review
match_df.to_csv('manual_mapping.csv', index=False)
print(f'Saved manual_mapping.csv — {len(match_df)} foods')
match_df

Saved manual_mapping.csv — 176 foods


,Food,matched_to,score
0,white flour,"Egg, white, raw, fresh",86
1,rice long grain,"Babyfood, dinner, turkey and rice, strained",86
2,spaghetti,"Babyfood, dinner, spaghetti and tomato and mea...",90
3,white bread,"Fast foods, submarine sandwich, cold cut on wh...",90
4,whole wheat bread,"Bread, pita, whole-wheat",95
...,...,...,...
171,"Raspberries, fresh","Egg, duck, whole, fresh, raw",86
172,"Raspberries, frozen","Raspberries, frozen, red, sweetened",90
173,"Strawberries, fresh","Egg, goose, whole, fresh, raw",86
174,"Strawberries, frozen","Strawberries, frozen, unsweetened",90


## Step 4 — Review Low Confidence Matches

Open `manual_mapping.csv` and fix any bad matches in the `matched_to` column.
Anything below score 75 likely needs a manual fix.

In [12]:
low = match_df[match_df['score'] < 75].sort_values('score')
print(f'Low confidence matches: {len(low)}')
low

Low confidence matches: 0


,Food,matched_to,score


In [13]:
# ── MANUAL FIXES ─────────────────────────────────────────────────────────────
# After reviewing low confidence matches above, add corrections here.
# To search for the right name: 
#   nutrients[nutrients['Ingredient description'].str.contains('egg', case=False)]
# ─────────────────────────────────────────────────────────────────────────────

manual_fixes = {

    # ── GRAINS ───────────────────────────────────────────────────────────────
    'white flour'               : 'Wheat flour, white, all-purpose, self-rising, enriched',
    'rice long grain '          : 'Rice, white, long-grain, regular, raw, enriched',
    'spaghetti '                : 'Spaghetti, dry, enriched',
    'white bread'               : 'Bread, white, commercially prepared',
    'whole wheat bread '        : 'Bread, whole-wheat, commercially prepared',
    'chocolate chip cookie '    : 'Cookies, chocolate chip, commercially prepared, regular',

    # ── MEATS ────────────────────────────────────────────────────────────────
    'ground chuck beef '        : 'Beef, ground, 80% lean meat / 20% fat, raw',
    'ground beef '              : 'Beef, ground, 85% lean meat / 15% fat, raw',
    'lean ground beef '         : 'Beef, ground, 95% lean meat / 5% fat, raw',
    'stew beef '                : 'Beef, chuck, arm pot roast, separable lean only, trimmed to 1/8" fat, choice, raw',
    'steak, round'              : 'Beef, round, top round, separable lean only, trimmed to 1/8" fat, choice, raw',
    'bacon'                     : 'Pork, cured, bacon, unprepared',
    'pork chops '               : 'Pork, fresh, loin, center loin (chops), bone-in, separable lean only, raw',
    'chicken legs, bone-in'     : 'Chicken, broilers or fryers, leg, meat and skin, raw',
    'chicken breast, boneless ' : 'Chicken, broilers or fryers, breast, meat only, raw',

    # ── DAIRY & EGGS ─────────────────────────────────────────────────────────
    'grade A eggs '             : 'Egg, whole, raw, fresh',
    'whole milk '               : 'Milk, whole, 3.25% milkfat, with added vitamin D',
    'american cheese '          : 'Cheese, pasteurized process, American, without added vitamin D',
    'ice cream, regular, bulk ' : 'Ice creams, vanilla',
    'low-fat milk'              : 'Milk, reduced fat, fluid, 2% milkfat, with added vitamin A and vitamin D',
    'yogurt, whole-fat plain '  : 'Yogurt, plain, whole milk, 8 grams protein per 8 ounce',
    'butter, stick '            : 'Butter, salted',

    # ── SWEETENERS & OTHER ───────────────────────────────────────────────────
    'white sugar '              : 'Sugars, granulated',
    'ground coffee '            : 'Beverages, coffee, brewed, prepared with tap water',
    'potato chip '              : 'Snacks, potato chips, plain, salted',

    # ── VEGETABLES ───────────────────────────────────────────────────────────
    'acorn squash '             : 'Squash, winter, acorn, raw',
    'artichoke, fresh '         : 'Artichokes, (globe or french), raw',
    'artichoke, canned'         : 'Artichokes, (globe or french), cooked, boiled, drained, without salt',
    'asparagus, fresh '         : 'Asparagus, raw',
    'asparagus, canned'         : 'Asparagus, canned, drained solids',
    'butternut squash'          : 'Squash, winter, butternut, raw',
    'green cabbage '            : 'Cabbage, raw',
    'red cabbage '              : 'Cabbage, red, raw',
    'carrots, whole, fresh '    : 'Carrots, raw',
    'celery '                   : 'Celery, raw',
    'collard greens, fresh '    : 'Collards, raw',
    'collard greens, frozen '   : 'Collards, frozen, unprepared',
    'corn, fresh '              : 'Corn, sweet, yellow, raw',
    'corn, canned '             : 'Corn, sweet, yellow, canned, whole kernel, drained solids',
    'corn, frozen '             : 'Corn, sweet, yellow, frozen, kernels cut off cob, unprepared',
    'Cucumbers, fresh '         : 'Cucumber, with peel, raw',
    'Great northern beans, canned' : 'Beans, great northern, canned',
    'Great northern beans, dried'  : 'Beans, great northern, mature seeds, raw',
    'Green beans, fresh'        : 'Beans, snap, green, raw',
    'Green beans, canned'       : 'Beans, snap, green, canned, regular pack, drained solids',
    'Green beans, frozen '      : 'Beans, snap, green, frozen, all styles, unprepared',
    'Green peas, canned'        : 'Peas, green, canned, drained solids',
    'Green peas, frozen '       : 'Peas, green, frozen, unprepared',
    'Green peppers'             : 'Peppers, sweet, green, raw',
    'Kale, fresh '              : 'Kale, raw',
    'Kidney beans, canned '     : 'Beans, kidney, red, canned',
    'Kidney beans, dried'       : 'Beans, kidney, all types, mature seeds, raw',
    'Lettuce, romaine, heads'   : 'Lettuce, cos or romaine, raw',
    'Lettuce, romaine, hearts'  : 'Lettuce, cos or romaine, raw',
    'Lima beans, canned'        : 'Lima beans, large, canned, drained solids',
    'Lima beans, frozen '       : 'Lima beans, large, frozen, unprepared',
    'Lima beans, dried'         : 'Lima beans, large, mature seeds, raw',
    'canned mixed vegetables: peas and carrots'                     : 'Peas and carrots, canned, regular pack, drained solids',
    'frozen mixed vegetables: peas and carrots'                     : 'Peas and carrots, frozen, unprepared',
    'frozen mixed vegetables: carrots, peas, corn, and green beans' : 'Vegetables, mixed, frozen, unprepared',
    'frozen mixed vegetables: broccoli, cauliflower, and carrots'   : 'Vegetables, mixed, frozen, unprepared',
    'Mushrooms, whole'          : 'Mushrooms, white, raw',
    'Mushrooms, pre-sliced'     : 'Mushrooms, white, raw',
    'Mustard greens, canned'    : 'Mustard greens, canned, drained solids',
    'Navy beans, canned'        : 'Beans, navy, canned',
    'Navy beans, dried'         : 'Beans, navy, mature seeds, raw',
    'Okra, fresh '              : 'Okra, raw',
    'Olives, canned'            : 'Olives, ripe, canned (small-extra large)',
    'Onions, fresh '            : 'Onions, raw',
    'Pinto beans, canned '      : 'Beans, pinto, mature seeds, canned',
    'Pinto beans, dried '       : 'Beans, pinto, mature seeds, raw',
    'Potatoes, fresh '          : 'Potatoes, flesh and skin, raw',
    'Radish'                    : 'Radishes, raw',
    'Red bell peppers'          : 'Peppers, sweet, red, raw',
    'Spinach, fresh '           : 'Spinach, raw',
    'Sweet potatoes, fresh '    : 'Sweet potato, raw, unprepared',
    'Tomatoes, grape and cherry': 'Tomatoes, red, ripe, raw, year round average',
    'Tomatoes, Roma and plum'   : 'Tomatoes, red, ripe, raw, year round average',
    'Tomatoes, large round'     : 'Tomatoes, red, ripe, raw, year round average',
    'Tomatoes, canned '         : 'Tomatoes, red, ripe, canned, packed in tomato juice',
    'Turnip greens, fresh '     : 'Turnip greens, raw',
    'Zucchini, fresh '          : 'Squash, summer, zucchini, includes skin, raw',
    'black beans, canned'       : 'Beans, black, canned',
    'black beans, dried'        : 'Beans, black, mature seeds, raw',
    'blackeye peas, canned'     : 'Cowpeas, common (blackeyes, crowder, southern), canned, plain',
    'blackeye peas, dried '     : 'Cowpeas, common (blackeyes), mature seeds, raw',
    'broccoli florets, fresh '  : 'Broccoli, raw',
    'broccoli heads, fresh '    : 'Broccoli, raw',

    # ── FRUITS ───────────────────────────────────────────────────────────────
    'Apples'                            : 'Apples, raw, with skin',
    'applesauce'                        : 'Applesauce, canned, unsweetened, without added ascorbic acid',
    'Apples, frozen juice concentrate'  : 'Apple juice, frozen concentrate, unsweetened, undiluted, without added ascorbic acid',
    'Apricots, fresh '                  : 'Apricots, raw',
    'Bananas'                           : 'Bananas, raw',
    'Berries, mixed, frozen '           : 'Blueberries, frozen, unsweetened',  # best available proxy
    'Blackberries, fresh '              : 'Blackberries, raw',
    'Blueberries, fresh '               : 'Blueberries, raw',
    'Cantaloupe, fresh '                : 'Melons, cantaloupe, raw',
    'Cherries, fresh '                  : 'Cherries, sweet, raw',
    'Clementines, fresh '               : 'Tangerines, (mandarin oranges), raw',
    'Dates, dried'                      : 'Dates, medjool',
    'Fruit cocktail, in juice'          : 'Fruit cocktail, (peach and pear and apricot and grape and cherry), canned, juice pack, solids and liquids',
    'Fruit cocktail, in syrup or water' : 'Fruit cocktail, (peach and pear and apricot and grape and cherry), canned, heavy syrup, solids and liquids',
    'Grapes, frozen juice concentrate'  : 'Grape juice, canned or bottled, unsweetened, with added ascorbic acid',
    'Mangoes, fresh '                   : 'Mangos, raw',
    'Mangoes, dried'                    : 'Apricots, dried, sulfured, uncooked',  # best available proxy
    'Orange, frozen juice concentrate'  : 'Orange juice, frozen concentrate, unsweetened, undiluted',
    'Papaya, fresh '                    : 'Papayas, raw',
    'Papaya, dried'                     : 'Apricots, dried, sulfured, uncooked',  # best available proxy
    'Peaches, fresh '                   : 'Peaches, raw',
    'Peach, canned in juice'            : 'Peaches, canned, juice pack, solids and liquids',
    'Peaches, canned in syrup '         : 'Peaches, canned, heavy syrup pack, solids and liquids',
    'Pears, fresh '                     : 'Pears, raw',
    'Pears, canned in syrup '           : 'Pears, canned, heavy syrup pack, solids and liquids',
    'Pineapple, fresh '                 : 'Pineapple, raw, all varieties',
    'Pineapple, canned in syrup'        : 'Pineapple, canned, heavy syrup pack, solids and liquids',
    'Pineapple, dried'                  : 'Apricots, dried, sulfured, uncooked',  # best available proxy
    'Pineapple, frozen juice concentrate': 'Pineapple juice, canned or bottled, unsweetened, without added ascorbic acid',
    'Plum, fresh '                      : 'Plums, raw',
    'Plum (prunes), dried'              : 'Plums, dried (prunes), uncooked',
    'Plum (prune) juice '               : 'Prune juice, canned',
    'Pomegranate, fresh '               : 'Pomegranates, raw',
    'Raspberries, fresh '               : 'Raspberries, raw',
    'Strawberries, fresh '              : 'Strawberries, raw',
}

print(f"Total manual fixes: {len(manual_fixes)}")

# Apply fixes
for food, fix in manual_fixes.items():
    match_df.loc[match_df['Food'] == food, 'matched_to'] = fix
    match_df.loc[match_df['Food'] == food, 'score'] = 100

print(f'Applied {len(manual_fixes)} manual fixes')

Total manual fixes: 123
Applied 123 manual fixes


In [14]:
# Helper: search for the right nutrient name
# Change the keyword to whatever food you're looking for
keyword = 'egg'
nutrients[nutrients['Ingredient description'].str.contains(keyword, case=False)][['ingred_code','Ingredient description']]

,ingred_code,Ingredient description
46,1057,Eggnog
76,1123,"Egg, whole, raw, fresh"
77,1124,"Egg, white, raw, fresh"
78,1125,"Egg, yolk, raw, fresh"
79,1128,"Egg, whole, cooked, fried"
80,1129,"Egg, whole, cooked, hard-boiled"
81,1131,"Egg, whole, cooked, poached"
82,1132,"Egg, whole, cooked, scrambled"
83,1133,"Egg, whole, dried"
84,1138,"Egg, duck, whole, fresh, raw"


## Step 5 — Merge Prices with Nutrients

In [15]:
# Add matched name to foods
foods['matched_name'] = foods['Food'].map(
    dict(zip(match_df['Food'], match_df['matched_to']))
)

# Merge
merged = foods.merge(
    nutrients,
    left_on='matched_name',
    right_on='Ingredient description',
    how='inner'
)

print(f'Successfully merged: {len(merged)} foods')
merged[['Food', 'Price', 'matched_name']].head(10)

Successfully merged: 137 foods


,Food,Price,matched_name
0,rice long grain,0.970,"Rice, white, long-grain, regular, raw, enriched"
1,whole wheat bread,2.451,"Bread, whole-wheat, commercially prepared"
2,ground chuck beef,4.644,"Beef, ground, 80% lean meat / 20% fat, raw"
3,ground beef,4.791,"Beef, ground, 85% lean meat / 15% fat, raw"
4,lean ground beef,6.393,"Beef, ground, 95% lean meat / 5% fat, raw"
5,stew beef,6.773,"Beef, chuck, arm pot roast, separable lean onl..."
6,"steak, sirloin",10.313,"Beef, flank, steak, separable lean only, trimm..."
7,bacon,6.808,"Pork, cured, bacon, unprepared"
8,ham,5.484,"Ham, chopped, canned"
9,grade A eggs,4.823,"Egg, whole, raw, fresh"


## Step 6 — Normalize Prices to Per-100g

In [16]:
unit_to_grams = {
    'pound'  : 453.592,
    'pound ' : 453.592,
    'oz'     : 28.3495,
    'gallon' : 3785.41,
    'egg'    : 50.0,
}

def price_per_100g(row):
    unit  = str(row['Units']).strip().lower()
    qty   = row['Quantity']
    price = row['Price']
    grams = unit_to_grams.get(unit, None)
    if grams is None:
        return None
    return (price / (qty * grams)) * 100

merged['price_per_100g'] = merged.apply(price_per_100g, axis=1)
merged = merged.dropna(subset=['price_per_100g'])
merged = merged.sort_values('price_per_100g').drop_duplicates(subset='Food')

print(f'Foods with valid prices: {len(merged)}')
merged[['Food', 'Units', 'Price', 'price_per_100g']].head(10)

Foods with valid prices: 137


,Food,Units,Price,price_per_100g
136,Watermelon,pound,0.405825,0.089469
16,low-fat milk,gallon,3.817000,0.100835
10,whole milk,gallon,4.204000,0.111058
87,Bananas,pound,0.609595,0.134393
121,"Pineapple, fresh",pound,0.668427,0.147363
93,"Cantaloupe, fresh",pound,0.791924,0.174590
31,green cabbage,pound,0.826600,0.182234
14,white sugar,pound,0.860000,0.189598
64,"Potatoes, fresh",pound,0.921544,0.203166
82,Apple juice,oz,0.951115,0.209685


## Step 7 — Build Matrix A and Price Vector p

In [17]:
non_nutrient_cols = ['Food', 'Quantity', 'Units', 'Price', 'Year', 'FCD_id',
                     'FDC_match', 'FDC_type', 'matched_name', 'ingred_code',
                     'Ingredient description', 'price_per_100g']

nutrient_cols = [c for c in merged.columns if c not in non_nutrient_cols]

merged = merged.set_index('Food')

# Matrix A: foods x nutrients
A = merged[nutrient_cols].fillna(0)

# Price vector p
p = merged['price_per_100g']

print('Matrix A shape (foods x nutrients):', A.shape)
print('Price vector p length:', len(p))
A

Matrix A shape (foods x nutrients): (137, 65)
Price vector p length: 137


,Capric acid,Lauric acid,Myristic acid,Palmitic acid,Palmitoleic acid,Stearic acid,Oleic acid,Linoleic Acid,Linolenic Acid,Stearidonic acid,...,Vitamin B12,"Vitamin B-12, added",Vitamin B6,Vitamin C,Vitamin D,Vitamin E,"Vitamin E, added",Vitamin K,Water,Zinc
Food,,,,,,,,,,,,,,,,,,,,,
Watermelon,0.001,0.001,0.000,0.008,0.000,0.006,0.037,0.050,0.000,0.0,...,0.00,0.0,0.045,8.1,0.0,0.05,0.0,0.1,91.45,0.10
low-fat milk,0.049,0.055,0.175,0.558,0.027,0.243,0.508,0.062,0.008,0.0,...,0.53,0.0,0.038,0.2,1.2,0.03,0.0,0.2,89.21,0.48
whole milk,0.075,0.077,0.297,0.829,0.000,0.365,0.812,0.120,0.075,0.0,...,0.45,0.0,0.036,0.0,1.3,0.07,0.0,0.3,88.13,0.37
Bananas,0.001,0.002,0.002,0.102,0.010,0.005,0.022,0.046,0.027,0.0,...,0.00,0.0,0.367,8.7,0.0,0.10,0.0,0.5,74.91,0.15
"Pineapple, fresh",0.000,0.000,0.000,0.005,0.001,0.003,0.012,0.023,0.017,0.0,...,0.00,0.0,0.112,47.8,0.0,0.02,0.0,0.7,86.00,0.12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Papaya, dried",0.000,0.000,0.000,0.017,0.000,0.000,0.074,0.074,0.000,0.0,...,0.00,0.0,0.143,1.0,0.0,4.33,0.0,3.1,30.89,0.39
"Apricots, dried",0.000,0.000,0.000,0.017,0.000,0.000,0.074,0.074,0.000,0.0,...,0.00,0.0,0.143,1.0,0.0,4.33,0.0,3.1,30.89,0.39
"Mangoes, dried",0.000,0.000,0.000,0.017,0.000,0.000,0.074,0.074,0.000,0.0,...,0.00,0.0,0.143,1.0,0.0,4.33,0.0,3.1,30.89,0.39


In [18]:
# Save outputs
A.to_csv('matrix_A.csv')
p.to_csv('price_vector_p.csv', header=True)
print('Saved matrix_A.csv and price_vector_p.csv')

Saved matrix_A.csv and price_vector_p.csv
